### LLM Chain 만들기


## 1. 환경 구성

### 1) 라이브러리 설치

In [1]:
# uv add python-dotenv langchain langchain-openai

### 2) GROQ 인증키 설정

In [2]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:3])

gsk


In [3]:
import langchain
print(langchain.__version__)

1.4.3


In [4]:
from langchain_openai import ChatOpenAI

# Groq(OpenAI 호환) 공통 설정 — 모델/온도 변경은 이곳만 수정
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트
GROQ_MODEL = "openai/gpt-oss-120b"  # 대안: "qwen/qwen3.8-27b"
#MOVIE_MODEL = "qwen/qwen3.8-27b"  # 영화 추천 예제용 
TEMPERATURE = 0.7


def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()
print(f'model name = {llm.model_name}')

model name = openai/gpt-oss-120b


## 2. LLM Chain

### 1) Prompt + LLM


In [5]:
# model: 공통 설정 셀의 llm 사용

# chain 실행
result = llm.invoke("인공지능 모델의 학습 원리에 대하여 쉽게 설명해 주세요.")
print(type(result))
print(result)

<class 'langchain_core.messages.ai.AIMessage'>
content='## 인공지능 모델이 배우는 원리 (쉽게 풀어쓴 설명)\n\n### 1. **데이터가 ‘교재’다**\n- **예시**: 여러분이 학교에서 수학을 배울 때 교과서와 문제집을 사용하죠?  \n- **AI**도 마찬가지로 **많은 데이터**(이미지, 텍스트, 음성 등)를 교재처럼 사용합니다. 이 데이터가 많을수록, 모델이 더 다양한 상황을 경험하게 돼요.\n\n### 2. **모델 = ‘학생’**\n- 모델은 **수많은 파라미터(가중치)** 로 이루어진 수학적인 함수입니다.  \n- 파라미터는 ‘학생의 머릿속에 저장된 지식’이라고 생각하면 됩니다. 처음엔 무작위값으로 시작해서, 학습을 거치면서 점점 바뀝니다.\n\n### 3. **목표(정답)와 ‘오답’**\n- **목표**: 주어진 입력(예: 사진)에서 원하는 출력(예: 사진에 있는 물건 이름)을 맞추는 것.  \n- **오답**: 모델이 내놓은 출력이 정답과 얼마나 차이가 나는지를 알려주는 **손실 함수(loss function)** 라는 척도. 손실이 클수록 ‘오답’이 많다는 뜻이죠.\n\n### 4. **학습 = ‘반복 연습’**\n1. **입력 데이터를 모델에 넣는다** → 모델이 현재 파라미터로 예측값을 만든다.  \n2. **예측값과 정답을 비교한다** → 손실 함수를 통해 “얼마나 틀렸는가”를 계산한다.  \n3. **오차를 줄이기 위해 파라미터를 조금씩 바꾼다** → 이때 쓰이는 것이 **경사 하강법(gradient descent)** 라는 수학적 방법.  \n4. **위 과정을 여러 번(에폭, epoch) 반복**한다. → 마치 문제를 풀고 틀린 부분을 교정하면서 실력을 키우는 과정과 비슷합니다.\n\n### 5. **‘학습률’이라는 조절 장치**\n- 파라미터를 바꾸는 폭을 **학습률(learning rate)** 라고 합니다.  \n- **너무 크게** 바꾸면 목표를 지나쳐 버리고, **너무 작게

In [6]:
print(result.content)

## 인공지능 모델이 배우는 원리 (쉽게 풀어쓴 설명)

### 1. **데이터가 ‘교재’다**
- **예시**: 여러분이 학교에서 수학을 배울 때 교과서와 문제집을 사용하죠?  
- **AI**도 마찬가지로 **많은 데이터**(이미지, 텍스트, 음성 등)를 교재처럼 사용합니다. 이 데이터가 많을수록, 모델이 더 다양한 상황을 경험하게 돼요.

### 2. **모델 = ‘학생’**
- 모델은 **수많은 파라미터(가중치)** 로 이루어진 수학적인 함수입니다.  
- 파라미터는 ‘학생의 머릿속에 저장된 지식’이라고 생각하면 됩니다. 처음엔 무작위값으로 시작해서, 학습을 거치면서 점점 바뀝니다.

### 3. **목표(정답)와 ‘오답’**
- **목표**: 주어진 입력(예: 사진)에서 원하는 출력(예: 사진에 있는 물건 이름)을 맞추는 것.  
- **오답**: 모델이 내놓은 출력이 정답과 얼마나 차이가 나는지를 알려주는 **손실 함수(loss function)** 라는 척도. 손실이 클수록 ‘오답’이 많다는 뜻이죠.

### 4. **학습 = ‘반복 연습’**
1. **입력 데이터를 모델에 넣는다** → 모델이 현재 파라미터로 예측값을 만든다.  
2. **예측값과 정답을 비교한다** → 손실 함수를 통해 “얼마나 틀렸는가”를 계산한다.  
3. **오차를 줄이기 위해 파라미터를 조금씩 바꾼다** → 이때 쓰이는 것이 **경사 하강법(gradient descent)** 라는 수학적 방법.  
4. **위 과정을 여러 번(에폭, epoch) 반복**한다. → 마치 문제를 풀고 틀린 부분을 교정하면서 실력을 키우는 과정과 비슷합니다.

### 5. **‘학습률’이라는 조절 장치**
- 파라미터를 바꾸는 폭을 **학습률(learning rate)** 라고 합니다.  
- **너무 크게** 바꾸면 목표를 지나쳐 버리고, **너무 작게** 바꾸면 배우는 속도가 느려집니다. 적절한 학습률을 찾는 것이 중요합니다.

### 6. **검증과 테스트**
- **검증 데이터**: 

### 2) PromptTemplate + LLM

In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("You are an expert in AI Expert. Answer the question.\
                                       <Question>: {input}에 대해 쉽게 설명해주세요.")

print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} template='You are an expert in AI Expert. Answer the question.                                       <Question>: {input}에 대해 쉽게 설명해주세요.'


In [8]:
# llm: 공통 설정 셀의 llm 사용
# chain 연결 (LCEL)
chain = prompt | llm
print(type(chain))

# chain 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.ai.AIMessage'>
content='### 인공지능 모델이 어떻게 “학습”하나요? – 쉬운 비유와 핵심 개념\n\n---\n\n## 1. 전체 흐름을 한 문장으로\n> **인공지능 모델은 ‘입력(데이터)’과 ‘정답(라벨)’을 가지고 ‘오답을 줄이는 방법(손실 함수와 최적화)’을 반복하면서 스스로 규칙을 찾아갑니다.**\n\n---\n\n## 2. 비유: “퍼즐 맞추기 게임”\n\n| 단계 | 퍼즐 비유 | AI 학습에서 실제 의미 |\n|------|----------|----------------------|\n| **준비** | 여러 조각(데이터)과 그림(정답)을 준비 | **데이터**(입력)와 **라벨**(정답) |\n| **시도** | 조각을 대충 맞춰보는 시도 | 모델이 **예측**(출력)을 만든다 |\n| **피드백** | “이 조각은 여기 안 맞아!” 라는 힌트 받음 | **손실 함수**가 현재 예측과 정답 사이 차이를 수치화 |\n| **수정** | 조각을 조금씩 옮겨가며 맞춤 | **최적화 알고리즘(예: 경사하강법)**이 파라미터(가중치)를 조금씩 바꿈 |\n| **반복** | 계속 맞출 때까지 반복 → 그림 완성 | **에포크(전체 데이터 한 번 순회)**를 여러 번 수행 → 손실이 최소화될 때까지 반복 |\n| **완성** | 퍼즐이 완성되면 더 이상 조정 필요 없음 | 모델이 **일반화**되어 새로운 데이터에도 좋은 성능을 보임 |\n\n---\n\n## 3. 핵심 요소 5가지\n\n| 요소 | 역할 | 쉬운 설명 |\n|------|------|-----------|\n| **데이터** | 학습에 쓰이는 재료 | 퍼즐 조각 |\n| **모델(구조)** | 예측을 만드는 기계 | 퍼즐을 맞추는 규칙(예: “가장 비슷한 모양끼리 맞춘다”) |\n| **손실 함수(Loss)** 

In [9]:
print(result.content)

### 인공지능 모델이 어떻게 “학습”하나요? – 쉬운 비유와 핵심 개념

---

## 1. 전체 흐름을 한 문장으로
> **인공지능 모델은 ‘입력(데이터)’과 ‘정답(라벨)’을 가지고 ‘오답을 줄이는 방법(손실 함수와 최적화)’을 반복하면서 스스로 규칙을 찾아갑니다.**

---

## 2. 비유: “퍼즐 맞추기 게임”

| 단계 | 퍼즐 비유 | AI 학습에서 실제 의미 |
|------|----------|----------------------|
| **준비** | 여러 조각(데이터)과 그림(정답)을 준비 | **데이터**(입력)와 **라벨**(정답) |
| **시도** | 조각을 대충 맞춰보는 시도 | 모델이 **예측**(출력)을 만든다 |
| **피드백** | “이 조각은 여기 안 맞아!” 라는 힌트 받음 | **손실 함수**가 현재 예측과 정답 사이 차이를 수치화 |
| **수정** | 조각을 조금씩 옮겨가며 맞춤 | **최적화 알고리즘(예: 경사하강법)**이 파라미터(가중치)를 조금씩 바꿈 |
| **반복** | 계속 맞출 때까지 반복 → 그림 완성 | **에포크(전체 데이터 한 번 순회)**를 여러 번 수행 → 손실이 최소화될 때까지 반복 |
| **완성** | 퍼즐이 완성되면 더 이상 조정 필요 없음 | 모델이 **일반화**되어 새로운 데이터에도 좋은 성능을 보임 |

---

## 3. 핵심 요소 5가지

| 요소 | 역할 | 쉬운 설명 |
|------|------|-----------|
| **데이터** | 학습에 쓰이는 재료 | 퍼즐 조각 |
| **모델(구조)** | 예측을 만드는 기계 | 퍼즐을 맞추는 규칙(예: “가장 비슷한 모양끼리 맞춘다”) |
| **손실 함수(Loss)** | 현재 예측이 얼마나 틀렸는지 점수 | “얼마나 틀렸는지 점수 매기기” |
| **최적화(Optimizer)** | 손실을 줄이기 위해 파라미터를 바꿈 | “점수를 낮추기 위해 조각을 조금씩 움직임” |
| **학습 과정(Iteration/Epoch)

### 3) PromptTemplate + LLM(invoke()) + StrOutputParser
* invoke() 함수는 동기(synchronous) 방식으로 호출함

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser
print(type(chain))

# 3. chain의 invoke 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.base.TextAccessor'>
### 인공지능 모델, 특히 **머신러닝/딥러닝 모델**이 어떻게 “학습”하는지 쉽게 풀어볼게요.

---

## 1. 학습이란 “경험을 통해 실수를 줄이는 과정”
- **우리 인간**이 새로운 기술을 배울 때, 처음엔 많이 틀리다가 **실수를 되돌아보고** 고쳐 나가면서 점점 잘하게 되죠.
- 인공지능도 **데이터(경험)** 를 가지고 **실수를 최소화**하도록 스스로 파라미터(뇌의 연결 강도)를 바꿉니다.

---

## 2. 기본 구성 요소

| 요소 | 역할 | 비유 |
|------|------|------|
| **데이터** | 모델이 보고 배울 입력‑출력 쌍 | 교과서·문제집 |
| **모델(네트워크)** | 입력을 받아서 예측을 만드는 함수 | 학생이 문제를 푸는 방법 |
| **손실 함수(Loss)** | 예측이 정답과 얼마나 차이나는지 수치화 | 시험 점수(오답 수) |
| **옵티마이저(Optimizer)** | 손실을 줄이기 위해 파라미터를 조정 | 공부 전략(공부량, 복습 주기) |
| **에포크(Epoch)** | 전체 데이터셋을 한 번 다 본 횟수 | 한 번의 전체 시험 연습 |

---

## 3. 학습 흐름 (한 사이클)

1. **입력 데이터를 모델에 넣는다**  
   → 예를 들어, 손글씨 사진(입력) → “이게 무엇인가?” 라는 질문.

2. **모델이 예측값을 만든다**  
   → “이 사진은 ‘3’이라고 추정”.

3. **손실 함수를 통해 오차를 계산한다**  
   → 실제 정답이 ‘5’라면, 오차가 크게 나오죠.

4. **옵티마이저가 파라미터를 조금씩 바꾼다**  
   → “내가 ‘3’이라고 판단한 이유가 뭐지?” 라는 질문에 답하려고, 가중치(연결 강도)를 미세하게 조정.

5. **다음 데이터에 대해 1~4 과정을

### 4) PromptTemplate + LLM(stream()) + StrOutputParser
* stream() 함수는 비동기(asynchronous) 방식으로 호출함

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

# chain 연결 (LCEL)
chain = prompt | llm | StrOutputParser()

# 스트리밍 출력을 위한 요청
answer = chain.stream({"input": "인공지능 모델의 학습 원리"})
# 스트리밍 출력
#print(answer)

for token in answer:
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

### 인공지능 모델, 특히 **머신러닝/딥러닝** 모델이 어떻게 학습하는지 쉽게 풀어볼게요.

---

## 1. 기본 아이디어: “경험을 통해 배우기”
- **사람**은 새로운 일을 할 때 예시(경험)를 보고 “이건 이렇게 하면 된다”는 규칙을 스스로 만들어요.  
- **AI 모델**도 비슷하게, **데이터(예시)**를 많이 보여 주면서 “입력 → 출력” 사이의 관계를 스스로 찾아냅니다.

---

## 2. 구성 요소

| 요소 | 설명 | 비유 |
|------|------|------|
| **데이터** | 모델에게 보여줄 입력과 정답(라벨) | 교과서와 정답지 |
| **모델(네트워크)** | 입력을 받아서 출력을 만들어 내는 수학식(함수) | 뇌의 신경망처럼 연결된 작은 계산기들 |
| **손실 함수(Loss)** | 모델이 만든 출력과 정답 사이의 차이를 숫자로 나타낸 것 | 시험 점수(오답이 많을수록 점수가 낮음) |
| **최적화 알고리즘** | 손실을 최소화하도록 모델 내부 파라미터를 바꾸는 방법 | 공부하면서 틀린 문제를 고쳐 나가는 과정 |
| **에폭(Epoch)** | 전체 데이터를 한 번 다 사용해 학습하는 과정 | 한 권의 교과서를 다 읽고 복습하는 한 사이클 |

---

## 3. 학습 흐름 (예시: 고양이 사진을 ‘고양이’라고 맞추는 모델)

1. **데이터 준비**  
   - **입력**: 고양이 사진, 강아지 사진 등 이미지  
   - **정답(라벨)**: “고양이”, “강아지”

2. **모델 초기화**  
   - 파라미터(가중치)를 무작위 값으로 설정 → 아직은 아무것도 모름

3. **한 번의 학습 단계(배치) 수행**  
   a. **예측**: 모델에 사진을 넣으면 현재 파라미터로 “고양이 확률 0.2” 같은 결과를 냄  
   b. **손실 계산**: 정답은 “고양이(1)”. 손실 함수는 “예측 0.2와 정답 1 사이 차이”를 수치화 → 큰 손실  
   c. **역전파(Backpropagation)**

##### 2) Multiple Chains
* Multi Chain을 활용한 영화 추천 및 줄거리 요약 

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
                                            줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")

# 영화 예제용 모델 (공통 설정 셀의 get_llm 사용)
llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
movie = chain1.invoke({"genre": "Drama"})  # 영화 제목 얻기

print(type(movie))
print(" ===> 추천된 영화:")  # movie 값 출력
print(movie)


<class 'langchain_core.messages.base.TextAccessor'>
 ===> 추천된 영화:
**추천 영화: <밀양 (Secret Sunshine)>, 2007**

- **감독**: 이창동  
- **주연**: 김혜수, 오정세, 정우성  
- **줄거리**: 남편과 아들을 사고로 잃은 후, 작은 해안 마을 ‘밀양’으로 이사 온 심혜원(김혜수). 슬픔과 절망 속에서도 새로운 삶을 찾으려 애쓰지만, 또 다른 비극이 찾아오면서 그녀는 믿음과 용서, 그리고 인간 관계의 복잡함을 마주하게 됩니다.  
- **추천 포인트**  
  1. **깊이 있는 감정 묘사** – 김혜수의 섬세하고 강렬한 연기가 관객을 감정의 소용돌이로 끌어들입니다.  
  2. **사회·인간에 대한 통찰** – 이창동 특유의 인간 중심적인 시선이 일상 속 작은 사건들을 통해 큰 질문을 던집니다.  
  3. **시각적 미학** – 해안가 풍경과 빗속 장면 등, 색채와 구도가 감정을 더욱 고조시킵니다.  
  4. **수상 경력** – 칸 영화제 심사위원대상·여우주연상(김혜수) 등 국제 영화제에서 다수 수상, 한국 영화의 위상을 높인 작품입니다.  

> **한 줄 요약**: “삶의 고통과 회복을 섬세하게 그린, 인간 존재의 의미를 묵직하게 탐구하는 감동적인 드라마.”  

만약 한국 드라마 영화를 처음 접하신다면, <밀양>은 감정선이 풍부하고 스토리가 깊어 오래도록 기억에 남을 것입니다. 감상 후에 느낀 감정을 차분히 정리해 보시면, 영화가 전하고자 하는 메시지를 더욱 깊게 체험하실 수 있을 거예요. 즐거운 감상 되세요! 🎬✨


In [13]:
# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
# prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
#                                             줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)
print(chain2)

# 실행: "SF" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:\n", response) 

first={
  movie: ChatPromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, template='{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.'), additional_kwargs={})])
         | ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000284E99E9040>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000284FFB26A50>, root_client=<openai.OpenAI object at 0x00000284FF0A48F0>, root_async_client=<openai.AsyncOpenAI object at 0x00000284FEE717F0>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', stream_chunk_timeout=120.0)
         | StrOutputParser()
} middle=[ChatPromptTemplate

##### chain1과 chain2에서 영화 제목이 일관되게 전달 되도록 변경 

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고, 줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요.")

llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: "Drama" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:")
pprint(response)


🔹 영화 줄거리 요약:
('버닝 (Burning)\n'
 '\n'
 '제목: 버닝  \n'
 '감독: 이창동  \n'
 '캐스팅: 이병헌, 스티븐 연, 전종서  \n'
 '줄거리: 청년 이성진(이병헌)은 고등학교 동창 선희(전종서)를 우연히 만나 다시 가까워진다. 선희는 오래전 떠난 남자친구 '
 '‘베르그’(스티븐 연)를 찾아오겠다고 약속한다. 베르그는 부유하고 신비로운 인물로, “뭔가를 태우고 있다”는 말만 남긴다. 선희가 사라진 '
 '뒤, 이성진은 베르그와 점점 얽히게 되고, 두 사람 사이에 숨겨진 ‘불’이 의미하는 바를 파헤치며 인간의 욕망·소외·불안을 마주하게 '
 '된다.')


In [17]:
# 모델 설정 (공통 설정 셀의 get_llm 사용)
model = get_llm(GROQ_MODEL)

# 멀티 체인 실행 함수
def movie_chain(genre):
    print(f"\n🔹 1단계: '{genre}' 장르 영화 추천")
    
    # 체인 1: 영화 추천
    response1 = model.invoke(f"{genre} 장르에서 유명한 영화 제목 하나만 알려줘")
    movie = response1.content.strip()
    print(f"   → 추천 영화: {movie}")
    
    print(f"\n 2단계: '{movie}' 영화 정보 조회")
    
    # 체인 2: 영화 정보
    response2 = model.invoke(f"{movie} 영화에 대해 간단히 알려줘. 줄거리, 감독, 출연진 포함해서")
    info = response2.content
    print(f"   → 영화 정보:\n{info}")
    
    return movie, info

# 실행
genre = input("원하는 장르를 입력하세요: ")
movie, info = movie_chain(genre)
print(f"\n 최종 결과: '{movie}' 영화가 추천되었습니다!")


🔹 1단계: 'action' 장르 영화 추천
   → 추천 영화: "다이 하드 (Die Hard)"는 액션 장르를 대표하는 매우 유명한 영화 중 하나입니다.

 2단계: '"다이 하드 (Die Hard)"는 액션 장르를 대표하는 매우 유명한 영화 중 하나입니다.' 영화 정보 조회
   → 영화 정보:
**다이 하드 (Die Hard, 1988)**  

| 구분 | 내용 |
|------|------|
| **감독** | **존 맥티어넌 (John McTiernan)** – 액션 영화와 스릴러 장르에서 다수의 명작을 만든 미국 감독. |
| **주연 배우** | • **브루스 윌리스 (Bruce Willis)** – 주인공 **존 매클레인 (John McClane)** 형사 <br>• **앨런 립리 (Alan Rickman)** – 악당 **한스 그루버 (Hans Gruber)** <br>• **Bonnie Bedelia** – 매클레인의 아내 **홀리 (Holly)** <br>• **Reginald VelJohnson** – 경찰관 **Al Powell** <br>• **Alexander Godunov** – 테러리스트 **Karl** 등 |
| **개봉** | 1988년 7월 15일 (미국) |
| **제작·배급** | 20th Century Fox (현재는 디즈니/폭스) |
| **장르** | 액션·스릴러·범죄 |

---

### 간단한 줄거리

미국 뉴욕 경찰관 **존 매클레인**은 크리스마스 이브에 LA에 있는 고층 빌딩 **Nakatomi Plaza**(Nakatomi 사옥)로 아내 **홀리**를 만나러 간다.  
그곳에서 Nakatomi 사는 연례 파티가 열리고, 매클레인은 파티에 잠시 끼어들어 **‘테러리스트’**라 자칭하는 **한스 그루버** 일당에게 포위된다.  

그들은 빌딩 안에 있는 1억 달러 규모의 국채를 강탈하려는 목적으로 고도로 조직된 인질극을 벌이며, 경찰과 FBI는 외부에서 상황을 통제하려 한다. 하지만 매클레인은 건물 안에